In [3]:
%pip install pyarrow fastparquet pandas scikit-learn matplotlib

   ---------------------------------------- 0.0/28.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/28.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/28.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/28.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/28.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/28.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/28.2 MB ? eta -:--:--
    --------------------------------------- 0.5/28.2 MB 266.1 kB/s eta 0:01:45
    --------------------------------------- 0.5/28.2 MB 266.1 kB/s eta 0:01:45
    --------------------------------------- 0.5/28.2 MB 266.1 kB/s eta 0:01:45
    --------------------------------------- 0.5/28.2 MB 266.1 kB/s eta 0:01:45
    --------------------------------------- 0.5/28.2 MB 266.1 kB/s eta 0:01:45
    --------------------------------------- 0.5/28.2 MB 266.1 kB/s eta 0:01:45
   - -------------------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd

# Wenn die Datei direkt neben dem Notebook liegt:
file_name = 'snapshots_2026-03-06.parquet' 

try:
    df = pd.read_parquet(file_name, engine = 'fastparquet')
    print("✅ Datei erfolgreich geladen!")
    print(f"Anzahl Zeilen: {len(df)}")
    print("-" * 30)
    print("Spalten im Dataset:", df.columns.tolist())
    print("-" * 30)
    display(df.head()) # Zeigt die ersten 5 Zeilen schön formatiert an
except Exception as e:
    print(f"❌ Fehler: {e}")
    print("Prüfe, ob der Dateiname exakt stimmt (inkl. .parquet)")

✅ Datei erfolgreich geladen!
Anzahl Zeilen: 3611054
------------------------------
Spalten im Dataset: ['timestamp_received', 'timestamp_created_at', 'market_id', 'update_type', 'data']
------------------------------


,timestamp_received,timestamp_created_at,market_id,update_type,data
0,1772755479569,1772755479636,0x0007deb167d0bb816e2e847a15435f3e384f97f9e3e2...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."
1,1772757306842,1772757306884,0x0007deb167d0bb816e2e847a15435f3e384f97f9e3e2...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."
2,1772758741660,1772758741701,0x0007deb167d0bb816e2e847a15435f3e384f97f9e3e2...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."
3,1772758749082,1772758749140,0x0007deb167d0bb816e2e847a15435f3e384f97f9e3e2...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."
4,1772757678461,1772757678492,0x0008043c3ed513ecff7ee64380fc943dc73eb3dfb667...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."


In [7]:
display(df.columns.tolist())

['timestamp_received',
 'timestamp_created_at',
 'market_id',
 'update_type',
 'data']

In [8]:
# Zeigt den Inhalt der ersten Zeile in der Spalte 'data'
print(df['data'].iloc[0])

{"update_type": "book_snapshot", "market_id": "0x0007deb167d0bb816e2e847a15435f3e384f97f9e3e252a4b66f2b64e5590b27", "token_id": "44386886286128895176476854069379354537282255501687744017982288947118429085822", "side": "YES", "best_bid": "0.07", "best_ask": "0.51", "timestamp": 1772755479.5698578, "bids": [["0.01", "206.52"], ["0.02", "5.35"], ["0.03", "98.94"], ["0.07", "55.99"]], "asks": [["0.99", "5460.16"], ["0.98", "200"], ["0.97", "5.84"], ["0.95", "50"], ["0.94", "220"], ["0.93", "10.33"], ["0.89", "100"], ["0.58", "1100"], ["0.57", "500"], ["0.56", "63"], ["0.53", "92.55"], ["0.52", "1000"], ["0.51", "37"]]}


In [1]:
import pandas as pd
import json
from sklearn.linear_model import LinearRegression

# 1. Schritt: Die Spalte 'data' von String in Dictionary umwandeln (falls nötig)
# Da du die Daten oben als JSON gepostet hast, müssen wir sie parsen
if isinstance(df['data'].iloc[0], str):
    df['data'] = df['data'].apply(json.loads)

# 2. Schritt: Wichtige Werte aus dem Dictionary extrahieren
df['bid'] = df['data'].apply(lambda x: float(x.get('best_bid', 0)))
df['ask'] = df['data'].apply(lambda x: float(x.get('best_ask', 0)))

# 3. Schritt: Unser 'Target' (y) berechnen: Der Mid-Price
df['mid_price'] = (df['bid'] + df['ask']) / 2

# 4. Schritt: Ein Feature bauen (X) -> Orderbook Imbalance
# Wir schauen, ob mehr Volumen bei den Bids (Käufern) oder Asks (Verkäufern) liegt
def get_total_volume(order_list):
    return sum([float(item[1]) for item in order_list])

df['bid_vol'] = df['data'].apply(lambda x: get_total_volume(x.get('bids', [])))
df['ask_vol'] = df['data'].apply(lambda x: get_total_volume(x.get('asks', [])))

# 5. ML-Modell: Sagt das Volumen den Preis voraus?
# Wir löschen Zeilen mit 0 (Fehler/Snapshots ohne Orders)
df_ml = df[df['mid_price'] > 0].copy()

X = df_ml[['bid_vol', 'ask_vol']]
y = df_ml['mid_price']

model = LinearRegression()
model.fit(X, y)

print(f"Modell trainiert! R² Score: {model.score(X, y):.4f}")

KeyboardInterrupt: 